In [1]:
from pathlib import Path
import numpy as np
import random
#random.seed(1)

In [2]:
def read_file(file_name):
    path = Path(file_name)
    if not path.exists():
        print("File not found")
        return None

    text = path.read_text(encoding="utf-8", errors="ignore")
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    return lines

In [3]:
lines = read_file("p_median_capacitated.txt")    
print(lines[:5])
print("Total lines:", len(lines))

['1 713', '50 5 120', '1 2 62 3', '2 80 25 14', '3 36 88 1']
Total lines: 1336


In [4]:
def parse_instances(lines):
    i = 0
    instances = []

    while i < len(lines):
        inst_id, best = map(int, lines[i].split())
        i += 1

        n, p, cap = map(int, lines[i].split())
        i += 1

        xs, ys, ds = [], [], []
        for _ in range(n):
            parts = lines[i].split()
            i += 1
            xs.append(float(parts[1]))
            ys.append(float(parts[2]))
            ds.append(float(parts[3]))

        inst = {
            "id": inst_id,
            "best": best,
            "n": n,
            "p": p,
            "cap": cap,
            "x": np.array(xs),
            "y": np.array(ys),
            "demand": np.array(ds),
        }

        instances.append(inst)

    return instances

In [5]:
instances = parse_instances(lines)
print(instances[0]["p"], instances[0]["cap"],instances[0]["demand"] )

5 120 [ 3. 14.  1. 14. 19.  2. 14.  6.  7.  6. 10. 18.  3.  6. 20.  4. 14. 11.
 19. 15. 15.  4. 13. 13.  5. 16.  3.  7. 14. 17.  3.  3. 12. 14. 20. 13.
 10.  9.  6. 18.  7. 20.  9.  1.  8.  5.  1.  7.  9.  2.]


In [6]:
def build_distance_matrix(inst):
    x = inst["x"]
    y = inst["y"]

    dx = x[:, None] - x[None, :]
    dy = y[:, None] - y[None, :]

    dist = np.sqrt(dx*dx + dy*dy)

    inst["dist"] = dist
    return inst


In [7]:
inst0 = build_distance_matrix(instances[0])

print(inst0["dist"].shape)
print(inst0["dist"][0][:5])

(50, 50)
[ 0.         86.33075929 42.80186912 67.42403132 54.64430437]


In [8]:
def create_random_chromosome(inst):
    n = inst["n"]
    p = inst["p"]

    chromosome = random.sample(range(n), p)
    return chromosome

In [9]:
chrom = create_random_chromosome(instances[0])
print(chrom)
print("length:", len(chrom))

[21, 23, 4, 16, 44]
length: 5


In [10]:
def compute_dispersion(inst, chromosome):
    dist = inst["dist"]

    min_d = float("inf")

    for i in range(len(chromosome)):
        for j in range(i+1, len(chromosome)):
            f1 = chromosome[i]
            f2 = chromosome[j]

            d = dist[f1][f2]

            if d < min_d:
                min_d = d

    return min_d

In [11]:
disp = compute_dispersion(instances[0], chrom)
print("dispersion =", disp)

dispersion = 2.23606797749979


In [12]:
def compute_service_distance(inst, chromosome):
    dist = inst["dist"]
    n = inst["n"]

    total_distance = 0.0

    for customer in range(n):

        min_d = float("inf")

        for median in chromosome:
            d = dist[customer][median]

            if d < min_d:
                min_d = d

        total_distance += min_d

    return total_distance

In [13]:
service = compute_service_distance(instances[0], chrom)
print("service distance =", service)

service distance = 1247.3079238559621


In [14]:
def evaluate_solution(inst, chromosome):

    service = compute_service_distance(inst, chromosome)
    dispersion = compute_dispersion(inst, chromosome)

    return {
        "service": service,
        "dispersion": dispersion
    }

In [15]:
result = evaluate_solution(instances[0], chrom)
print(result)

{'service': np.float64(1247.3079238559621), 'dispersion': np.float64(2.23606797749979)}


In [16]:
def init_population(inst, pop_size):
    pop = []
    for _ in range(pop_size):
        chrom = create_random_chromosome(inst)
        fit = evaluate_solution(inst, chrom)
        pop.append({
            "chrom": chrom,
            "service": float(fit["service"]),
            "dispersion": float(fit["dispersion"]),
        })
    return pop

In [17]:
pop = init_population(instances[0], pop_size=20)
print("Population size:", len(pop))
print("First individual:", pop[0])

Population size: 20
First individual: {'chrom': [1, 5, 11, 34, 6], 'service': 1943.938318952648, 'dispersion': 7.0710678118654755}


In [18]:
def dominates(a, b):
    """
    a and b are of population
    """

    better_or_equal_service = a["service"] <= b["service"]
    better_or_equal_disp = a["dispersion"] >= b["dispersion"]

    strictly_better = (
        a["service"] < b["service"] or
        a["dispersion"] > b["dispersion"]
    )

    return better_or_equal_service and better_or_equal_disp and strictly_better

In [19]:
A = {"service":900, "dispersion":30}
B = {"service":950, "dispersion":25}

print(dominates(A,B)) 
print(dominates(B,A)) 

True
False


In [20]:
def non_dominated_sort(pop):
    S = {}          # S[p] = mängden lösningar som p dominerar
    n_dom = {}      # n_dom[p] = hur många som dominerar p
    fronts = [[]]   # fronts[0] = front 1

    for p in pop:
        S[id(p)] = []
        n_dom[id(p)] = 0

        for q in pop:
            if p is q:
                continue
            if dominates(p, q):
                S[id(p)].append(q)
            elif dominates(q, p):
                n_dom[id(p)] += 1

        if n_dom[id(p)] == 0:
            fronts[0].append(p)

    i = 0
    while i < len(fronts) and len(fronts[i]) > 0:
        next_front = []
        for p in fronts[i]:
            for q in S[id(p)]:
                n_dom[id(q)] -= 1
                if n_dom[id(q)] == 0:
                    next_front.append(q)
        i += 1
        if next_front:
            fronts.append(next_front)

    return fronts

In [21]:
fronts = non_dominated_sort(pop)
print("Antal fronter:", len(fronts))
print("Front 1 size:", len(fronts[0]))
print("Front 2 size:", len(fronts[1]) if len(fronts) > 1 else 0)

Antal fronter: 8
Front 1 size: 3
Front 2 size: 2
